# AiSynth Labs · AI DRUG guide · Notebook 2
## Machine learning

**Author:** Toluwanimi Odunewu (@toluwanimiso)  
**Last updated:** 2026-10-30  
**Runtime:** about 3 to 6 minutes · **GPU needed:** no  
**Projects covered:** 7 to 10 (predict, new chemistry, new years, uncertainty)  
**Licence:** CC BY 4.0

Part of the free "Get started in AI drug discovery" guide from AiSynth Labs. Run the cells from top to bottom. Change one thing at a time and write down what moved.

### Aim of this notebook
Train a model on real ChEMBL data, then try to break it: test it on chemistry it has not seen, on papers from later years, and ask how sure it is. You will finish with an honest number for how far to trust it.

**Runs on its own:** it rebuilds the dataset from Notebook 1 in the first cell, so you do not need to run Notebook 1 first.

### References
1. Zdrazil B et al. (2023). *The ChEMBL Database in 2023.* Nucleic Acids Res. [doi:10.1093/nar/gkad1004](https://doi.org/10.1093/nar/gkad1004).
2. Pedregosa F et al. (2011). *Scikit-learn: machine learning in Python.* J Mach Learn Res 12:2825-2830 (random forest, SVM, neural networks, cross-validation).
3. Butina D (1999). *Unsupervised data base clustering based on Daylight's fingerprint and Tanimoto similarity.* J Chem Inf Comput Sci 39:747-750 [doi:10.1021/ci9803381](https://doi.org/10.1021/ci9803381).
4. RDKit: open-source cheminformatics, [rdkit.org](https://www.rdkit.org).

**Note on project 10:** the ensemble here uses scikit-learn networks. The idea (many models, spread as uncertainty, calibrate on held-out data) is the same with PyTorch.

---
# Practical

In [ ]:
# --- SETUP: runs on Colab, Kaggle or on your own computer -------------------------------
import os, sys, random, importlib.util
ON_COLAB = 'google.colab' in sys.modules
ON_KAGGLE = os.path.exists('/kaggle/input')
if (ON_COLAB or ON_KAGGLE) and importlib.util.find_spec('rdkit') is None:
    print('Installing RDKit (about a minute) ...')
    !pip install -q rdkit
SEED = 42                      # fixed seed: you and everyone else should get the same numbers
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
print(f'Colab: {ON_COLAB} | Kaggle: {ON_KAGGLE} | seed {SEED}')

In [ ]:
# --- IMPORTS, SETTINGS AND THE READY LIBRARY (the same steps as Notebook 1, projects 3 and 4) --------------
import json, time, urllib.parse, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, Lipinski, rdFingerprintGenerator
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.ML.Cluster import Butina
RDLogger.DisableLog('rdApp.*')

# >>> Change these lines to work on another protein. CDK5 is CHEMBL4036, BACE1 is CHEMBL4822.
TARGET_NAME = 'ACHE'
TARGET_CHEMBL_ID = 'CHEMBL220'          # human acetylcholinesterase
ACTIVE_CUTOFF = 6.3                     # pIC50 at or above this counts as "active". It is a choice: say yours when you post.
PIC50_RANGE = (2, 12)                   # outside this range (IC50 below 1 picomolar or above 10 millimolar) a value is treated as an error and dropped
results = {}                            # every number you will want to post is saved here

CHEMBL = 'https://www.ebi.ac.uk/chembl/api/data/'

def get_json(url, tries=4):
    for k in range(tries):
        try:
            return json.load(urllib.request.urlopen(url, timeout=90))
        except Exception:
            time.sleep(2 + 3 * k)
    raise RuntimeError('could not reach ' + url)

def fetch_ic50(chembl_id):
    rows, offset = [], 0
    while True:
        q = urllib.parse.urlencode({'target_chembl_id': chembl_id, 'standard_type': 'IC50', 'standard_relation': '=',
                                    'assay_type': 'B', 'standard_units': 'nM', 'limit': 1000, 'offset': offset})
        page = get_json(CHEMBL + 'activity.json?' + q)
        rows += page['activities']
        if not page['page_meta']['next']:
            return pd.DataFrame(rows)
        offset += 1000

def rule_of_five_ok(m):
    ok = [Descriptors.ExactMolWt(m) <= 500, Lipinski.NumHAcceptors(m) <= 10, Lipinski.NumHDonors(m) <= 5, Descriptors.MolLogP(m) <= 5]
    return sum(ok) >= 3

def catalog(name):
    p = FilterCatalogParams(); p.AddCatalog(getattr(FilterCatalogParams.FilterCatalogs, name)); return FilterCatalog(p)
PAINS, BRENK = catalog('PAINS'), catalog('BRENK')

def build_library(chembl_id, apply_filters=True):
    """ChEMBL IC50 records -> one row per molecule (median pIC50) -> rule of five and PAINS filters."""
    raw = fetch_ic50(chembl_id)
    d = raw[['molecule_chembl_id', 'canonical_smiles', 'standard_value', 'document_year']].copy()
    d['standard_value'] = pd.to_numeric(d['standard_value'], errors='coerce')
    d = d.dropna(subset=['standard_value', 'canonical_smiles']); d = d[d['standard_value'] > 0]
    d['pIC50'] = 9 - np.log10(d['standard_value'])
    d = (d.groupby('molecule_chembl_id', as_index=False)
           .agg(smiles=('canonical_smiles', 'first'), pIC50=('pIC50', 'median'), document_year=('document_year', 'min')))
    d = d[d.pIC50.between(*PIC50_RANGE)]                  # drop the impossible potencies (see Notebook 1, project 3)
    d['mol'] = [Chem.MolFromSmiles(s) for s in d['smiles']]
    d = d[d['mol'].notna()].reset_index(drop=True)
    n_all = len(d)
    if apply_filters:
        d = d[[rule_of_five_ok(m) for m in d.mol]]
        d = d[[not PAINS.HasMatch(m) for m in d.mol]]
    return d.reset_index(drop=True), n_all

lib, n_all = build_library(TARGET_CHEMBL_ID)
print(f'{TARGET_NAME} ({TARGET_CHEMBL_ID}): {n_all:,} molecules in ChEMBL, {len(lib):,} left after the rule of five and PAINS')
results['library'] = dict(target=TARGET_NAME, molecules_all=n_all, molecules_library=len(lib))

## Project 7. Predict active or inactive
A fingerprint turns each molecule into 2048 yes/no bits. A molecule is called **active** if its pIC50 is at or above the cut-off. AUC measures how well a model ranks actives above inactives: 0.5 is a coin flip, 1.0 is perfect. The spread is across three folds, not a confidence interval.

In [ ]:
# --- PROJECT 7: predict active or inactive ---------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score

morgan3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)     # a molecule -> 2048 bits
X = np.array([morgan3.GetFingerprintAsNumPy(m) for m in lib.mol], dtype=np.uint8)
y = (lib.pIC50.values >= ACTIVE_CUTOFF).astype(int)                            # 1 = active
print(f'{len(y):,} molecules, {y.mean():.0%} active at pIC50 >= {ACTIVE_CUTOFF}')

MODELS = {'random forest': lambda: RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
          'SVM': lambda: SVC(kernel='rbf', C=1.0, gamma='scale'),
          'neural network': lambda: MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=200, random_state=SEED)}

def score_of(model, Xte):
    return model.predict_proba(Xte)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(Xte)

def cv_auc(make, splits):
    out = []
    for tr, te in splits:
        if len(set(y[te])) < 2 or len(set(y[tr])) < 2:
            continue
        m = make(); m.fit(X[tr], y[tr]); out.append(roc_auc_score(y[te], score_of(m, X[te])))
    return out

random_splits = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X, y))
p7 = {}
for name, make in MODELS.items():
    a = cv_auc(make, random_splits)
    p7[name] = dict(auc=round(float(np.mean(a)), 3), spread=round(float(np.std(a)), 3))
    print(f'{name:<16} AUC {np.mean(a):.3f}  (spread across the 3 folds: {np.std(a):.3f})')
results['p7'] = dict(molecules=len(y), active_share=round(float(y.mean()), 2), models=p7)

## Project 8. Does it work on new chemistry?
A random split puts close cousins of every test molecule in the training set, which flatters the model. Here whole clusters go to one side. Then we group the test molecules by how similar they are to anything in training: that is your trust rule.

In [ ]:
# --- PROJECT 8: does it work on new chemistry? ---------------------------------------------------------------
fp_rdkit = rdFingerprintGenerator.GetRDKitFPGenerator(maxPath=5, fpSize=2048)
fps = [fp_rdkit.GetFingerprint(m) for m in lib.mol]
dists = []
for i in range(1, len(fps)):
    dists.extend([1 - s for s in DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])])
cl = Butina.ClusterData(dists, len(fps), 0.2, isDistData=True)                  # the same clusters as Notebook 1, project 5
group = np.zeros(len(fps), dtype=int)
for k, c in enumerate(cl):
    group[list(c)] = k

cluster_splits = list(GroupKFold(3).split(X, y, groups=group))                 # a whole cluster is in train OR test, never both
a = cv_auc(MODELS['random forest'], cluster_splits)
print(f"random forest: random split {p7['random forest']['auc']:.3f}  ->  cluster split {np.mean(a):.3f}  (drop {p7['random forest']['auc'] - np.mean(a):.3f}; folds {[round(v, 3) for v in a]})")

# A trust rule: how close must a new molecule be to what the model has seen?
sims, scores, truth = [], [], []
fp_m2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
m2 = [fp_m2.GetFingerprint(m) for m in lib.mol]
for tr, te in cluster_splits:
    mdl = MODELS['random forest'](); mdl.fit(X[tr], y[tr])
    scores += list(mdl.predict_proba(X[te])[:, 1]); truth += list(y[te])
    train_fps = [m2[i] for i in tr]
    sims += [max(DataStructs.BulkTanimotoSimilarity(m2[i], train_fps)) for i in te]
sims, scores, truth = map(np.array, (sims, scores, truth))
rule = []
for lo, hi in [(0.0, 0.4), (0.4, 0.6), (0.6, 1.01)]:
    sel = (sims >= lo) & (sims < hi)
    ok = sel.sum() >= 20 and len(set(truth[sel])) == 2
    auc = roc_auc_score(truth[sel], scores[sel]) if ok else None
    rule.append(dict(nearest_training_similarity=f'{lo:.1f} to {min(hi, 1):.1f}', n=int(sel.sum()), auc=None if auc is None else round(float(auc), 3)))
    print(f'  nearest training molecule {lo:.1f} to {min(hi, 1):.1f}: {sel.sum():>5,} test molecules, AUC {auc if auc is None else round(auc, 3)}')
results['p8'] = dict(clusters=len(cl), random_auc=p7['random forest']['auc'], cluster_auc=round(float(np.mean(a)), 3),
                     cluster_fold_aucs=[round(float(v), 3) for v in a], trust_rule=rule)

## Project 9. A time-based reality check
The honest question is: trained on what is known today, how does it do on tomorrow's molecules? ChEMBL records the year of each paper, so we can train on older papers and test on newer ones.

In [ ]:
# --- PROJECT 9: a time-based reality check -----------------------------------------------------------------
yr = lib.document_year.fillna(-1).values
rows = []
for Y in range(2005, 2026):                            # train on papers older than year Y, test on papers from year Y
    tr, te = np.where((yr > 0) & (yr < Y))[0], np.where(yr == Y)[0]
    if len(te) < 30 or len(tr) < 100 or y[te].sum() < 5 or (1 - y[te]).sum() < 5:
        continue
    m = MODELS['random forest'](); m.fit(X[tr], y[tr])
    rows.append(dict(test_year=Y, n_train=len(tr), n_test=len(te), auc=round(float(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1])), 3)))
by_year = pd.DataFrame(rows)

# one single cut: the oldest 70 percent of papers train, the newest 30 percent test
order = np.argsort(yr, kind='stable'); order = order[yr[order] > 0]
cut_year = int(yr[order[int(0.7 * len(order))]])
tr, te = np.where((yr > 0) & (yr <= cut_year))[0], np.where(yr > cut_year)[0]
m = MODELS['random forest'](); m.fit(X[tr], y[tr])
single = roc_auc_score(y[te], m.predict_proba(X[te])[:, 1])
print(f'trained on papers up to {cut_year} ({len(tr):,} molecules), tested on the {len(te):,} after: AUC {single:.3f}  (random split: {p7["random forest"]["auc"]:.3f})')

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(by_year.test_year, by_year.auc, 'o-', color='black'); ax.axhline(p7['random forest']['auc'], color='#c9b800', lw=3)
ax.set_xlabel('test year'); ax.set_ylabel('AUC'); ax.set_title('yellow line = the random-split score'); plt.show()
recent = by_year[by_year.test_year >= 2015].auc
results['p9'] = dict(cut_year=cut_year, n_train=len(tr), n_test=len(te), auc=round(float(single), 3),
                     from_2015_min=round(float(recent.min()), 2), from_2015_max=round(float(recent.max()), 2), years=by_year.to_dict('records'))

## Project 10. How sure is each prediction?
Twenty networks, each started differently, will agree on molecules like the ones they have seen and disagree on strange ones. The disagreement is the uncertainty. We scale it on a held-out third, then check it on a different third.

In [ ]:
# --- PROJECT 10: how sure is each prediction? ----------------------------------------------------------------
from scipy.stats import spearmanr
rng = np.random.RandomState(SEED)
idx = rng.permutation(len(lib)); n_tr, n_cal = int(0.6 * len(lib)), int(0.2 * len(lib))
tr, cal, te = idx[:n_tr], idx[n_tr:n_tr + n_cal], idx[n_tr + n_cal:]
pic = lib.pIC50.values

preds = []
for s in range(20):                                   # 20 networks, each started from a different random seed
    net = MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=300, early_stopping=True, random_state=SEED + s)
    net.fit(X[tr], pic[tr]); preds.append(net.predict(X))
P = np.array(preds); mu, sd = P.mean(0), P.std(0)       # prediction = the average; uncertainty = how much the 20 disagree

k = float(np.median(np.abs(mu[cal] - pic[cal])) / np.median(sd[cal]))   # calibration on a held-out third: scale the spread to the typical error
err, unc = np.abs(mu[te] - pic[te]), sd[te] * k
order = np.argsort(unc); fifth = len(order) // 5
print(f'mean error, all test molecules:         {err.mean():.2f} pIC50 units')
print(f'mean error, most confident fifth:       {err[order[:fifth]].mean():.2f}')
print(f'mean error, least confident fifth:      {err[order[-fifth:]].mean():.2f}')
print(f'rank correlation of uncertainty with error: {spearmanr(unc, err).correlation:.2f}')
cover = float(np.mean(err <= 1.96 * unc))
print(f'a 95 percent interval (1.96 x uncertainty) held the true value {cover:.0%} of the time')

pot = [j for j in np.argsort(-mu[te]) if mu[te][j] >= 7.0]
best = sorted(pot, key=lambda j: unc[j])[:10]
top = pd.DataFrame([dict(molecule=lib.molecule_chembl_id[te[j]], predicted=round(float(mu[te][j]), 2), uncertainty=round(float(unc[j]), 2),
                          measured=round(float(pic[te][j]), 2)) for j in best])
display(top)
results['p10'] = dict(n_train=len(tr), n_cal=len(cal), n_test=len(te), mae_all=round(float(err.mean()), 2), mae_confident=round(float(err[order[:fifth]].mean()), 2),
                      mae_uncertain=round(float(err[order[-fifth:]].mean()), 2), spearman=round(float(spearmanr(unc, err).correlation), 3),
                      coverage_95=round(cover, 3), within_1_unit=int((abs(top.predicted - top.measured) <= 1.0).sum()),
                      worst=top.assign(e=(top.predicted - top.measured).abs()).sort_values('e').iloc[-1][['molecule', 'predicted', 'measured']].to_dict())

---
# Your turn
**Task:** change `ACTIVE_CUTOFF` to 7.0 and run everything again. **Expected output:** a smaller active share, and AUCs that move. Write which of the four numbers moved the most.

Then change `TARGET_CHEMBL_ID` to `CHEMBL4036` (CDK5) and look at the spread across folds.

In [ ]:
print(json.dumps(results, indent=1, default=str)[:3500])

---
# Discussion
**What you learned:** a score on a random split is the most flattering number a model can have. The same model drops on new chemistry and drops again on newer papers.

**What this cannot do, and where it breaks:** one target, one fingerprint, one active cut-off. The spread shown is across three folds, not a confidence interval. The uncertainty here was calibrated on one held-out third and still undercovers: a 95 percent interval did not hold 95 percent of the time. Small datasets (a few hundred molecules) give unstable folds. Date every number you post.